In [1]:
import numpy as np
import qutip as qt
import scipy.sparse as sp
import matplotlib.pyplot as plt

# Import custom modules
from quantum_gates import GATE_DICTIONARY
from circuit_engine import apply_instruction
from generator import generate_random_circuit
from visualize import draw_ascii_circuit

print("All modules loaded successfully! Ready to test.")

All modules loaded successfully! Ready to test.


In [5]:
# Cell 2: Generate a test circuit
num_qubits = 7
num_layers = 10

print(f"--- Generating a {num_qubits}-Qubit Circuit with {num_layers} Layers ---\n")

# Call your generator (Make sure it returns ops, descs, AND instructions)
ops, descs, instructions = generate_random_circuit(num_qubits, num_layers)

# Print out the English descriptions of each layer
for i, desc in enumerate(descs):
    print(f"Layer {i+1}: {desc}")

# ---------------------------------------------------------
# check EVERY layer for shape and unitarity
# ---------------------------------------------------------
for i, op in enumerate(ops):
    # This will crash the program immediately if any layer is the wrong shape
    expected_shape = (2**num_qubits, 2**num_qubits)
    assert op.shape == expected_shape, f"Error: Layer {i+1} shape is {op.shape}, expected {expected_shape}"
    
    # This will crash the program if any layer is not unitary
    assert op.isunitary, f"Error: Layer {i+1} is not unitary!"

print("\nSuccess! All layers are the correct shape and perfectly unitary.")

--- Generating a 7-Qubit Circuit with 10 Layers ---

Layer 1: 1-qubit layer: S(0), Z(1), H(2), S(3), S(4), Z(5), H(6)
Layer 2: 2-qubit layer: C-X(control=0, target=1)
Layer 3: 2-qubit layer: C-Z(control=4, target=2)
Layer 4: 2-qubit layer: C-H(control=3, target=5)
Layer 5: 1-qubit layer: T(0), Z(1), T(2), S(3), Y(4), S(5), X(6)
Layer 6: 2-qubit layer: C-Y(control=1, target=5)
Layer 7: 1-qubit layer: Y(0), H(1), Z(2), Y(3), Y(4), H(5), X(6)
Layer 8: 2-qubit layer: ISWAP(targets=[5, 2])
Layer 9: 1-qubit layer: X(0), T(1), H(2), X(3), Z(4), X(5), H(6)
Layer 10: 1-qubit layer: T(0), Z(1), X(2), Z(3), S(4), S(5), S(6)

Success! All layers are the correct shape and perfectly unitary.


In [6]:
# Draw the circuit
draw_ascii_circuit(num_qubits, instructions)


=== Generated Quantum Circuit ===
q0: ───[S]─────■───────────────────[T]───────────[Y]───────────[X]────[T]───
q1: ───[Z]────[X]──────────────────[Z]─────■─────[H]───────────[T]────[Z]───
q2: ───[H]───────────[Z]───────────[T]─────│─────[Z]─────x─────[H]────[X]───
q3: ───[S]────────────│──────■─────[S]─────│─────[Y]─────│─────[X]────[Z]───
q4: ───[S]────────────■──────│─────[Y]─────│─────[Y]─────│─────[Z]────[S]───
q5: ───[Z]──────────────────[H]────[S]────[Y]────[H]─────x─────[X]────[S]───
q6: ───[H]─────────────────────────[X]───────────[X]───────────[H]────[S]───



In [7]:
# Apply the circuit to a state
# Create the initial state |0000>
ket0 = qt.basis(2, 0)
initial_state = qt.tensor([ket0 for _ in range(num_qubits)])

print("Initial State (First 5 amplitudes):")
print(initial_state.full().flatten()[:5]) 

# apply the circuit using Dense matrices
current_state = initial_state.full()
for layer_op in ops:
    current_state = layer_op.full() @ current_state

print("\nFinal State Vector after Circuit (First 5 amplitudes):")
print(np.round(current_state.flatten()[:5], 3))

# Verify probabilities sum to 1
total_probability = np.sum(np.abs(current_state)**2)
print(f"\nTotal Probability: {total_probability:.5f} (Should be exactly 1.0)")

Initial State (First 5 amplitudes):
[1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]

Final State Vector after Circuit (First 5 amplitudes):
[1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]

Total Probability: 1.00000 (Should be exactly 1.0)
